In [1]:
import os
import re
import math
import torch
import soundfile as sf
import numpy as np
import torchaudio
import torch.nn as nn
from tqdm import tqdm
import torch.optim as optim
from dataclasses import dataclass
from datasets import load_dataset
from jiwer import wer
import pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split


c:\Users\jfvdk\Desktop\AISD\Git\Deep-Learning\SignWave-Models-transcription\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CSV_PATH = r"C:\Users\jfvdk\Desktop\Git\Deep-Learning\SignWave-Models-transcription\data\dataset.csv"   
AUDIO_COL = "audio_path"                    # column name  csv
TEXT_COL  = "transcription"                    # column name  csv
EPOCHS = 10
BATCH_SIZE = 16
LR = 1e-3
SAMPLE_RATE = 16000
N_MELS = 80
TRAIN_RATIO = 0.9   # 90% train, 10% eval
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
import torch

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


2.9.1+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 1
GPU name: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [65]:
ALLOWED = "abcdefghijklmnopqrstuvwxyz' "  # space included

def normalize_text(t: str) -> str:
    t = str(t).lower()
    # replace everything not allowed with space
    t = "".join(ch if ch in ALLOWED else " " for ch in t)
    # collapse spaces
    t = re.sub(r"\s+", " ", t).strip()
    return t


In [66]:
class SpecAugment(nn.Module):
    def __init__(self, freq_mask=15, time_mask=35, num_freq_masks=2, num_time_masks=2):
        super().__init__()
        self.freq_mask = freq_mask
        self.time_mask = time_mask
        self.num_freq_masks = num_freq_masks
        self.num_time_masks = num_time_masks

    def forward(self, x):
        # x: (T, F)
        T, F = x.shape

        # frequency masks
        for _ in range(self.num_freq_masks):
            f = torch.randint(0, self.freq_mask + 1, (1,)).item()
            f0 = torch.randint(0, max(1, F - f), (1,)).item()
            x[:, f0:f0+f] = 0

        # time masks
        for _ in range(self.num_time_masks):
            t = torch.randint(0, self.time_mask + 1, (1,)).item()
            t0 = torch.randint(0, max(1, T - t), (1,)).item()
            x[t0:t0+t, :] = 0

        return x


In [67]:
def build_vocab(texts):
    chars = set()
    for t in texts:
        t = normalize_text(t)
        chars.update(list(t))
    vocab = ["<blank>"] + sorted(list(chars))
    stoi = {ch: i for i, ch in enumerate(vocab)}
    itos = {i: ch for ch, i in stoi.items()}
    return vocab, stoi, itos

In [68]:
def text_to_ids(text, stoi):
    text = normalize_text(text)
    ids = [stoi[ch] for ch in text if ch in stoi and ch != "<blank>"]
    return torch.tensor(ids, dtype=torch.long)

In [80]:
class FeatureExtractor(nn.Module):
    def __init__(self, sample_rate=16000, n_mels=80):
        super().__init__()
        self.sample_rate = sample_rate
        self.melspec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate, n_mels=n_mels
        )
        self.to_db = torchaudio.transforms.AmplitudeToDB()

    def forward(self, wav, sr):
        # wav: (C,T) or (T,)
        if wav.dim() == 2:
            wav = wav.mean(dim=0)  # mono -> (T,)
        if sr != self.sample_rate:
            wav = torchaudio.functional.resample(wav, sr, self.sample_rate)

        feat = self.to_db(self.melspec(wav))  # (n_mels, frames)
        feat = feat.transpose(0, 1)           # (frames, n_mels)
        return feat

In [81]:
class CTCASR(nn.Module):
    def __init__(self, n_mels=80, hidden=256, layers=3, vocab_size=30, dropout=0.1):
        super().__init__()
        self.rnn = nn.LSTM(
            input_size=n_mels,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden * 2, vocab_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        logits = self.fc(out)  # (B, T, vocab)
        return logits

In [82]:
class CSVASRDataset(Dataset):
    def __init__(self, csv_path, audio_col="audio_path", text_col="transcription"):  
        self.df = pd.read_csv(csv_path)
        if audio_col not in self.df.columns or text_col not in self.df.columns:
            raise ValueError(f"CSV must have columns: {audio_col}, {text_col}")

        # Drop rows with missing values
        self.df = self.df.dropna(subset=[audio_col, text_col]).reset_index(drop=True)

        self.audio_col = audio_col
        self.text_col = text_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        audio_path = str(self.df.loc[idx, self.audio_col])
        text = str(self.df.loc[idx, self.text_col])


        audio, sr = sf.read(audio_path, dtype="float32")  # (T,) or (T,C)

        # convert to mono
        if audio.ndim == 2:
          audio = audio.mean(axis=1)

        wav = torch.from_numpy(audio)  # (T,)
        return {"wav": wav, "sr": sr, "text": text} 



In [83]:
@dataclass
class Batch:
    feats: torch.Tensor
    feat_lens: torch.Tensor
    targets: torch.Tensor
    target_lens: torch.Tensor

def pad_2d(seqs, pad_value=0.0):
    T_max = max(s.size(0) for s in seqs)
    F = seqs[0].size(1)
    out = torch.full((len(seqs), T_max, F), pad_value, dtype=torch.float32)
    lens = torch.tensor([s.size(0) for s in seqs], dtype=torch.long)
    for i, s in enumerate(seqs):
        out[i, : s.size(0), :] = s
    return out, lens

def make_collate(feat_extractor, stoi):
    def collate_fn(items):
        feats = []
        targets_list = []
        target_lens = []

        for it in items:
            feat = feat_extractor(it["wav"], it["sr"])
            feats.append(feat)

            ids = text_to_ids(it["text"], stoi)
            targets_list.append(ids)
            target_lens.append(ids.numel())

        feats_padded, feat_lens = pad_2d(feats, pad_value=0.0)

        target_lens = torch.tensor(target_lens, dtype=torch.long)
        targets = torch.cat(targets_list, dim=0) if len(targets_list) else torch.empty((0,), dtype=torch.long)

        return Batch(feats=feats_padded, feat_lens=feat_lens, targets=targets, target_lens=target_lens)
    return collate_fn

In [84]:
def greedy_decode(logits, itos, blank_id=0):
    pred = logits.argmax(dim=-1)  # (B, T)
    texts = []
    for seq in pred:
        prev = None
        out_chars = []
        for p in seq.tolist():
            if p != blank_id and p != prev:
                out_chars.append(itos[p])
            prev = p
        texts.append("".join(out_chars).replace("  ", " ").strip())
    return texts

In [85]:
def train_one_epoch(model, loader, loss_fn, opt):
    model.train()
    total = 0.0

    for batch in loader:
        x = batch.feats.to(DEVICE)
        feat_lens = batch.feat_lens.to(DEVICE)
        targets = batch.targets.to(DEVICE)
        target_lens = batch.target_lens.to(DEVICE)

        logits = model(x)
        log_probs = logits.log_softmax(dim=-1).transpose(0, 1)  # (T,B,V)

        loss = loss_fn(log_probs, targets, feat_lens, target_lens)

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()

        total += loss.item()

    return total / max(1, len(loader))


In [86]:
@torch.no_grad()
def evaluate_wer(model, loader, itos, blank_id=0):
    model.eval()
    refs, hyps = [], []

    for batch in loader:
        x = batch.feats.to(DEVICE)
        logits = model(x)
        pred_texts = greedy_decode(logits.cpu(), itos, blank_id=blank_id)

        offset = 0
        ref_texts = []
        for L in batch.target_lens.tolist():
            ids = batch.targets[offset:offset+L].tolist()
            offset += L
            ref_texts.append("".join(itos[i] for i in ids).strip())

        refs.extend(ref_texts)
        hyps.extend(pred_texts)

    return wer(refs, hyps), refs[:5], hyps[:5]


In [87]:
def main():
    torch.manual_seed(SEED)

    dataset = CSVASRDataset(CSV_PATH, AUDIO_COL, TEXT_COL)

    # Build vocab from ALL texts (or only training texts after split)
    all_texts = dataset.df[TEXT_COL].tolist()
    vocab, stoi, itos = build_vocab(all_texts)
    blank_id = 0
    print("Vocab size:", len(vocab))

    # Split train/eval
    n_total = len(dataset)
    n_train = int(TRAIN_RATIO * n_total)
    n_eval = n_total - n_train
    train_ds, eval_ds = random_split(dataset, [n_train, n_eval])

    feat_extractor = FeatureExtractor(sample_rate=SAMPLE_RATE, n_mels=N_MELS)
    collate_fn = make_collate(feat_extractor, stoi)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    eval_loader  = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    model = CTCASR(n_mels=N_MELS, hidden=256, layers=3, vocab_size=len(vocab), dropout=0.1).to(DEVICE)
    loss_fn = nn.CTCLoss(blank=blank_id, zero_infinity=True)
    opt = optim.Adam(model.parameters(), lr=LR)

    epoch_bar = tqdm(range(1, EPOCHS + 1), desc="Training Progress")

    for epoch in epoch_bar:
        tr_loss = train_one_epoch(model, train_loader, loss_fn, opt)
        w, refs5, hyps5 = evaluate_wer(model, eval_loader, itos, blank_id=blank_id)

        epoch_bar.set_postfix({
        "loss": f"{tr_loss:.4f}",
        "WER": f"{w:.4f}"
    })


        print(f"\nEpoch {epoch}/{EPOCHS}")
        print(f"Train loss: {tr_loss:.4f}")
        print(f"Eval WER : {w:.4f}")
        for r, h in zip(refs5, hyps5):
            print("REF:", r)
            print("HYP:", h)
            print("---")

    os.makedirs("checkpoints", exist_ok=True)
    torch.save({"model": model.state_dict(), "vocab": vocab}, "checkpoints/ctc_asr_csv.pt")
    print("\nSaved: checkpoints/ctc_asr_csv.pt")
    ckpt = {
    "model_state": model.state_dict(),
    "vocab": vocab,              # list of tokens, vocab[0] = "<blank>"
    "n_mels": N_MELS,
    "sample_rate": SAMPLE_RATE,
    "hidden": 256,
    "layers": 3,
}
    os.makedirs("checkpoints", exist_ok=True)
    torch.save(ckpt, "checkpoints/ctc_asr.pt")
    print("Saved checkpoints/ctc_asr.pt")

if __name__ == "__main__":
    main()

Vocab size: 29


Training Progress:   0%|          | 0/10 [01:15<?, ?it/s]


KeyboardInterrupt: 